# ※ 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.  

- 채점 기준:  
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과와 일치하는 경우에만 정답으로 인정됩니다.  
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# 14주차 과제

---
### Finetuning 과제

In [6]:
# 실습 환경 세팅

!pip install -q transformers datasets peft torch

In [7]:
# 공통 실행 환경 (모든 문제 공통))

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

In [8]:
# 개인이 발급받은 huggingface token 사용해주세요!
# 사용하고자 하는 모델에 대해서는 huggingface model card에서 access를 요청하셔야 사용가능합니다 (최초 1회만 필요)

from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 문제 1. pretrained model & baseline 추론 & Instruction Dataset 구성 및 전처리

# 문제 1-1. Tokenizer 및 모델 로드
# 사전학습 언어모델(Llama-3.2-1B-Instruct)과 토크나이저를 로드하시오.

# TODO
model_name =

tokenizer =
tokenizer.pad_token = tokenizer.eos_token

model =

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
# 문제 1-2. Fine-tuning 이전 Baseline 추론
# Fine-tuning 이전 모델의 응답을 생성하시오.

prompt = "Explain fine-tuning in simple terms."
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    # TODO
    outputs =

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explain fine-tuning in simple terms. Fine-tuning refers to the process of adjusting the parameters of a model to optimize its performance. In other words, it's a way of tweaking the settings to get the best possible results.

Think of


In [ ]:
# 문제 1-3. Instruction / Response 포맷 구성

dataset = load_dataset("tatsu-lab/alpaca", split="train[:200]")

def format_example(example):
    # TODO
    text =
    return {"text": text}

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
# 문제 1-4. Tokenization + Label 생성
# Causal LM 학습을 위한 전처리를 완성하시오.

def preprocess(example):
    # TODO
    tokenized =

    # TODO
    tokenized["labels"] =

    return tokenized

tokenized_dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# 문제 2. LoRA 기반 Fine-tuning 구성
# 문제 2-1. LoRA 설정 정의
# 다음 조건을 만족하도록 LoRA 설정을 구성하시오.

# rank = 8
# attention projection(q_proj, v_proj)에만 적용
# dropout 사용


# TODO
lora_config =


In [ ]:
# 문제 2-2. LoRA 적용 및 Trainable Parameter 확인

# TODO
lora_model =
lora_model.print_trainable_parameters()


trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
# 문제 2-3. Trainer 구성
# 다음 조건을 만족하도록 Trainer를 구성하시오.

# batch size = 2
# epoch = 1
# checkpoint 저장 없음
# 외부 로깅 없음

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# TODO
training_args =

# TODO
trainer =

In [ ]:
# 문제 2-4. Fine-tuning 실행

# TODO

Step,Training Loss


TrainOutput(global_step=50, training_loss=2.2378834533691405, metrics={'train_runtime': 53.8118, 'train_samples_per_second': 3.717, 'train_steps_per_second': 0.929, 'total_flos': 149606105088000.0, 'train_loss': 2.2378834533691405, 'epoch': 1.0})

In [ ]:
# 문제 3. Fine-tuning 결과 추론 및 모델 재사용
# 문제 3-1. Fine-tuned 모델 추론

prompt = "### Instruction:\nExplain LoRA.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt")

device = lora_model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    # TODO
    outputs =

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


### Instruction:
Explain LoRA.

### Response:
LoRA (Low Power Radio) is a technique used in wireless communication systems to conserve power. It is a variation of the traditional radio frequency (RF) technology used in wireless devices. In LoRA, the radio signal is transmitted at a lower power level, which reduces the energy consumption of the device.

LoRA has several benefits, including


In [ ]:
# 문제 3-2. LoRA 모델 저장

save_path = "./final_lora_model"

# TODO




In [ ]:
# 문제 3-3. 모델 재로드 후 추론

# TODO
reloaded_model =

device = inputs["input_ids"].device
reloaded_model = reloaded_model.to(device)

with torch.no_grad():
    outputs = reloaded_model.generate(**inputs, max_length=80)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


### Instruction:
Explain LoRA.

### Response:
LoRA stands for Low Power Radio. It is a technology used in wireless communication systems to conserve energy while transmitting data. LoRA is used in applications such as IoT devices, smart home devices, and wearables. It allows devices to transmit data while keeping their power consumption low, making them suitable for battery-powered devices. LoRA is designed


In [ ]:
# 문제 4. Full Fine-Tuning vs LoRA 비교
# 문제 4-1. Full Fine-Tuning 모델 파라미터 수 계산

# TODO
full_ft_model =

device = inputs["input_ids"].device
full_ft_model = full_ft_model.to(device)

total_params = sum(p.numel() for p in full_ft_model.parameters())
trainable_params = sum(
    p.numel() for p in full_ft_model.parameters() if p.requires_grad
)

print(trainable_params, total_params)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

1235814400 1235814400


In [ ]:
# 문제 4-2. LoRA 모델 파라미터 수 계산

# TODO
lora_total =
lora_trainable =

In [ ]:
# 문제 4-3. 동일 프롬프트 추론 비교

# TODO
with torch.no_grad():
    out_full =
    out_lora =



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
# 문제 5. Multi-LoRA & Forgetting 완화
# 문제 5-1. Task A용 LoRA Adapter 생성

# TODO
lora_config_A =

# TODO
model_A =

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 문제 5-2. Task B용 LoRA Adapter 생성

# TODO
lora_config_B =

# TODO
model_B =


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 문제 5-3. Adapter 교체 기반 추론

prompt = "Explain fine-tuning briefly."
inputs = tokenizer(prompt, return_tensors="pt")

# TODO
with torch.no_grad():
    out_A =
    out_B =



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


---
### Multimodal LLM 과제

In [26]:
! pip install -q transformers torch

In [27]:
# 공통 실습 환경 (모든 문제 공통)

import torch
import torch.nn as nn
from PIL import Image

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    CLIPVisionModel,
    CLIPImageProcessor
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 문제 6

# 다음 조건을 만족하도록 코드를 완성하시오.
# 입력 이미지를 CLIP processor로 변환
# Vision Encoder forward 수행
# 출력 tensor의 shape 출력
# GPT 계열 tokenizer를 사용해 padding 문제 없이 text를 tokenization 하시오

image = Image.new("RGB", (224, 224), color="white")

processor = CLIPImageProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)
vision_model = CLIPVisionModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

#===============================
# TODO 1: image → tensor 변환
inputs =

# TODO 2: vision encoder forward
with torch.no_grad():
    outputs =

#===============================

vision_embeds = outputs.last_hidden_state
print("Vision embedding shape:", vision_embeds.shape)


tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Describe the image in detail."

#===============================

# TODO 3: padding token 설정


# TODO 4: tokenizer 적용
inputs =

#===============================

print(inputs["input_ids"].shape)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.final_layer_norm.weight                       

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 7])


In [ ]:
# 문제 7

# Vision embedding을 language hidden size로 변환하는 projection layer를 정의하시오.

vision_dim = 768
language_dim = 768


#===============================
# TODO 1: projection layer 정의
vision_to_text =

#===============================

dummy_vision = torch.randn(1, 10, vision_dim)
projected = vision_to_text(dummy_vision)

print(projected.shape)


torch.Size([1, 10, 768])


In [ ]:
# 문제 8

# Vision embedding과 Text embedding을 sequence dimension으로 concat하시오.

batch_size = 1
vision_tokens = 10
text_tokens = 6
hidden_dim = 768

vision_embeds = torch.randn(batch_size, vision_tokens, hidden_dim)
text_embeds = torch.randn(batch_size, text_tokens, hidden_dim)

#===============================
# TODO: concat 수행
multimodal_embeds =


#===============================

print(multimodal_embeds.shape)


torch.Size([1, 16, 768])


In [ ]:
# 문제 9

# 아래 조건을 만족하는 forward()를 완성하시오.

# CLIP Vision Encoder 사용
# Vision → Language projection
# Text embedding과 concat
# Language Model에 inputs_embeds 전달

class MiniLLaVAModel(nn.Module):
    def __init__(self, vision_model, lm, tokenizer):
        super().__init__()
        self.vision_model = vision_model
        self.lm = lm
        self.tokenizer = tokenizer
        self.proj = nn.Linear(768, lm.config.hidden_size)

    def forward(self, pixel_values, input_ids, labels=None):
        #===============================

        # TODO 1: vision embedding
        vision_outputs =

        # TODO 2: projection
        vision_embeds =

        # TODO 3: text embedding
        text_embeds =

        # TODO 4: concat
        inputs_embeds =

        #===============================

        return self.lm(
            inputs_embeds=inputs_embeds,
            labels=labels
        )




In [ ]:
# 문제 10

# 다음 조건을 만족하도록 training step을 완성하시오.

# Vision Encoder freeze
# Projection + LM만 학습
# Loss는 language modeling loss 사용


# tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# models
vision_model = CLIPVisionModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

lm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
lm.config.pad_token_id = lm.config.eos_token_id

model = MiniLLaVAModel(
    vision_model=vision_model,
    lm=lm,
    tokenizer=tokenizer
).to(device)

model.train()
#===============================
# TODO 1: vision encoder freeze

#===============================

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

# dummy input
batch_size = 1
pixel_values = torch.randn(batch_size, 3, 224, 224).to(device)

text = "Describe the image."
text_ids = tokenizer(
    text,
    return_tensors="pt"
).input_ids.to(device)

# vision token count
with torch.no_grad():
    v = model.vision_model(pixel_values=pixel_values)
num_vision_tokens = v.last_hidden_state.size(1)

# 🔑 labels: vision + first text token은 -100
labels = torch.cat(
    [
        torch.full(
            (batch_size, num_vision_tokens + 1),
            -100,
            device=device
        ),
        text_ids[:, 1:]
    ],
    dim=1
)

# forward
outputs = model(
    pixel_values=pixel_values,
    input_ids=text_ids,
    labels=labels
)

#===============================
# TODO 2: loss 정의
loss =

#===============================

loss.backward()
optimizer.step()

print("✅ Training step completed. Loss:", loss.item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.final_layer_norm.weight                       

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


✅ Training step completed. Loss: 4.273977279663086
